In [1]:
from transformers import PreTrainedModel, PretrainedConfig, AutoModelForCausalLM, AutoModel, AutoProcessor, AutoTokenizer
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from transformers.modeling_outputs import CausalLMOutputWithPast
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import os
import json
from PIL import Image
from typing import List, Dict, Any

In [2]:
class VLMConfig(PretrainedConfig):
    model_type = "vlm_model"
    def __init__(self,
                 llm_model_path = "Qwen2.5-0.5B-Instruct",
                 vision_model_path = "siglip-so400m-patch14-384",
                 freeze_vision_model = True,
                 image_pad_num = 49,
                 **kwargs):
        self.llm_model_path = llm_model_path
        self.vision_model_path = vision_model_path
        self.freeze_vision_model = freeze_vision_model
        self.image_pad_num = image_pad_num
        super().__init__(**kwargs)

In [3]:
class VLM(PreTrainedModel):
    config_class = VLMConfig
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.llm_model = AutoModelForCausalLM.from_pretrained(self.config.llm_model_path)
        self.vision_model = AutoModel.from_pretrained(self.config.vision_model_path)
        self.processor = AutoProcessor.from_pretrained(self.config.llm_model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.llm_model_path)
        # 视觉向文本对其
        self.linear1 = nn.Linear(self.vision_model.config.vision_config.hidden_size * 4, self.llm_model.config.hidden_size)
        self.linear2 = nn.Linear(self.llm_model.config.hidden_size, self.llm_model.config.hidden_size)
        if self.config.freeze_vision_model: # 冻结两个模型的权重，在预训练过程中只训练两个用于对齐的全连接层的权重
            for param in self.vision_model.parameters():
                param.requires_grad = False
        for param in self.llm_model.parameters():
            param.requires_grad = False

    def forward(self, input_ids, labels, pixel_values, attention_mask=None):
        text_embeds = self.llm_model.get_input_embedding()(input_ids)

        image_embeds = self.vision_model.vision_model(pixel_values).last_hidden_state
        b, s, d = image_embeds.shape
        image_embeds = image_embeds.view(b, -1, d*4) # (b, 196, d) -> (b, 49, d*4) 压缩图片 token
        image_features = self.linear2(F.silu(self.linear1(image_embeds)))

        text_embeds = text_embeds.to(image_features.dtype)

        input_embeds = self.merge_input_ids_with_image_features(image_features, text_embeds, input_ids)
        outputs = self.llm_model(input_embeds=input_embeds, attention_mask=attention_mask)
        logits = outputs[0]
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id)
            loss = loss_fn(
                logits.view(-1, logits.size(-1)), labels.view(-1).to(logits.device)
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def merge_input_ids_with_image_features(self, image_features, inputs_embeds, input_ids):
        num_images, num_image_patches, embed_dim = image_features.shape
        batch_indices, image_indices = torch.where(input_ids == self.tokenizer('<|image_pad|>')['input_ids'][0])

        inputs_embeds[batch_indices, image_indices] = image_features.view(-1, embed_dim)
        return inputs_embeds

In [4]:
class MyDataset(Dataset):
    def __init__(self, image_path, data_path, tokenizer, processor, config):
        self.image_path = image_path
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.processor = processor
        self.config = config
        with open(self.data_path, 'r', encoding='utf-8') as f:
            self.datas = json.load(f)

    def __len__(self):
        return len(self.datas)

    def __getitem__(self, index):
        sample = self.datas[index]
        try:
            image_name = sample['image']
            conversations = sample['conversations']
            q_text = self.tokenizer.apply_chat_template(
                [{"role":"system", "content":'You are a helpful assistant.'}, {"role":"user", "content":conversations[0]['value']}],
                tokenize=False,
                add_generation_prompt=True
            ).replace('<image>', '<|image_pad|>'*self.config.image_pad_num)
            a_text = conversations[1]['value'] + self.tokenizer.eos_token
            q_input_idx = self.tokenizer(q_text)['index_ids']
            a_input_idx = self.tokenizer(a_text)['index_idx']
            input_idx = q_input_idx + a_input_idx
            labels = [self.tokenizer.pad_token_id] * len(q_input_idx) + a_input_idx
            input_idx = input_idx[:-1]
            labels = labels[1:]

            image = Image.open(os.path.join(self.image_path, image_name)).convert('RGB')
            pixel_values = self.processor(text=None, images=image)['pixel_values']
        except:
            default_image = Image.new('RGB', (224, 224), color='white')
            pixel_values = self.processor(text=None, images=default_image)['pixel_values']
            q_text = self.tokenizer.apply_chat_template(
                [{"role":"system", "content":'You are a helpful assistant.'}, {"role":"user", "content":"图片内容是什么\n<image>"}],
                tokenize=False,
                add_generation_prompt=True
            ).replace('<image>', '<|image_pad|>'*self.config.image_pad_num)
            a_text = '图片内容为空' + self.tokenizer.eos_token
            q_input_ids = self.tokenizer(q_text)['input_ids']
            a_input_ids = self.tokenizer(a_text)['input_ids']
            input_ids = q_input_ids + a_input_ids
            labels = [self.tokenizer.pad_token_id] * len(q_input_ids) + a_input_ids
            input_ids = input_ids[:-1]
            labels = labels[1:]
        return {
            'input_ids': input_ids,
            'labels': labels,
            'pixel_values': pixel_values
        }

In [5]:
class MyDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        max_len = max(len(feature['input_ids']) for feature in features)
        input_ids = []
        labels = []
        pixel_values = []
        for feature in features:
            input_ids.append(feature['input_ids'] + [self.tokenizer.pad_token_id] * (max_len - len(feature['input_ids'])))
            labels.append(feature['labels'] + [self.tokenizer.pad_token_id] * (max_len - len(feature['labels'])))
            pixel_values.append(feature['pixel_values'])

        return {'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
                'pixel_values': torch.cat(pixel_values, dim=0)}

In [7]:
config = VLMConfig()
model = VLM(config)
print(model)
print(f'model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')
images_path = "LLaVA-CC3M-Pretrain-595K/images"
data_path = "Chinese-LLaVA-Vision-Instructions/LLaVA-CC3M-Pretrain-595K/chat-translated.json"
tokenizer = AutoTokenizer.from_pretrained(config.llm_model_path)
processor = AutoTokenizer.from_pretrained(config.vision_model_path)
output_dir = "/save"
args = TrainingArguments(
    output_dir=output_dir,
    do_train=True,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    num_train_epochs=5,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    gradient_accumulation_steps=8,
    logging_steps=100,
    report_to='tensorboard',
    dataloader_pin_memory=True,
    dataloader_num_workers=0
)
trainer = Trainer(
    model = model,
    args=args,
    train_dataset=MyDataset(images_path, data_path, tokenizer, processor, config),
    data_collator=MyDataCollator(tokenizer)
)

trainer.train(resume_from_checkpoint=False)
trainer.save_model('/save')
trainer.save_state()

VLM(
  (llm_model): Qwen2ForCausalLM(
    (model): Qwen2Model(
      (embed_tokens): Embedding(151936, 896)
      (layers): ModuleList(
        (0-23): 24 x Qwen2DecoderLayer(
          (self_attn): Qwen2SdpaAttention(
            (q_proj): Linear(in_features=896, out_features=896, bias=True)
            (k_proj): Linear(in_features=896, out_features=128, bias=True)
            (v_proj): Linear(in_features=896, out_features=128, bias=True)
            (o_proj): Linear(in_features=896, out_features=896, bias=False)
            (rotary_emb): Qwen2RotaryEmbedding()
          )
          (mlp): Qwen2MLP(
            (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
            (up_proj): Linear(in_features=896, out_features=4864, bias=False)
            (down_proj): Linear(in_features=4864, out_features=896, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
          (post_attention_layernorm): Qwen2RMSNorm((

ValueError: fp16 mixed precision requires a GPU (not 'mps').